Configurando ambiente de trabalho

In [0]:
%sql
use catalog stack_overgol;

In [0]:
from pyspark.sql.functions import (
    col, lower, trim, when, regexp_extract, regexp_replace,
    length, concat, lit, split, explode, substring
)

Avaliações

In [0]:
df = spark.table("bronze.avaliacoes")

df = df.withColumn(
    "recomenda",
    when(
        lower(trim(col("recomenda"))).isin("s", "sim", "yes", "1"),
        True
    ).when(
        lower(trim(col("recomenda"))).isin("n", "nao", "não", "no", "0"),
        False
    ).otherwise(None)
)

df = df.select( 
    col("id_avaliacao").cast("varchar"), 
    col("id_pedido").cast("varchar"),
    col("id_cliente").cast("varchar"),
    col("id_produto").cast("varchar"),
    col("nota_produto").cast("int"), # tem conversões a serem feitas
    col("comentario").cast("varchar").alias("comentario_avaliacao"),
    col("nota_nps").cast("int"), # tem conversões a serem feitas
    col("recomenda").alias("recomenda_produto"), # tem conversões a serem feitas
    col("data_avaliacao").cast("timestamp")) # tem conversões a serem feitas
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.avaliacoes")

Catálogos Produto

In [0]:
df = spark.table("bronze.catalogo_produtos")

df = df.withColumn(
    "ativo",
    when(
        lower(trim(col("ativo"))).isin("s", "sim", "yes", "1"),
        True
    ).when(
        lower(trim(col("ativo"))).isin("n", "nao", "não", "no", "0"),
        False
    ).otherwise(None)
)

df = df.select( 
    col("id_produto").cast("varchar"), 
    col("nome_produto").cast("varchar"),
    col("categoria").cast("varchar").alias("categoria_produto"), # tem conversões a serem feitas
    col("preco").cast("numeric, 2").alias("preco_produto"), # tem conversões a serem feitas
    col("fornecedor").cast("varchar").alias("fornecedor_produto"),
    col("peso_kg").cast("numeric, 2").alias("peso_kg_produto"), # tem conversões a serem feitas
    col("estoque_disponivel").cast("int").alias("estoque_produto"), # tem conversões a serem feitas
    col("avaliacao_interna").cast("numeric, 2"), # tem conversões a serem feitas
    col("ativo").cast("boolean").alias("produto_ativo"),
    col("data_cadastro_produto").cast("date")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.catalogo_produtos")

Trilha de ações do usuário

In [0]:
df = spark.table("bronze.clickstream")

df = df.select( 
    col("id_evento").cast("varchar"), 
    col("id_sessao").cast("varchar"),
    col("id_cliente").cast("varchar"), 
    col("id_dispositivo").cast("varchar"), 
    col("id_produto").cast("varchar"),
    col("tipo_evento").cast("varchar"), # tem conversões a serem feitas
    col("canal").cast("varchar"), # tem conversões a serem feitas
    col("dispositivo").cast("varchar"), # tem conversões a serem feitas
    col("origem_sessao").cast("varchar"),
    col("data_evento").cast("timestamp"),
    col("tempo_pagina_seg").cast("int") # tem conversões a serem feitas
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clickstream")

Clientes

In [0]:
df = spark.table("bronze.clientes")

df = df.withColumn(
    "origem",
    when(lower(trim(col("origem"))).isin("web"), "Web")
    .when(lower(trim(col("origem"))).isin("app"), "App")
    .when(lower(trim(col("origem"))).isin("indicação", "indicacao"), "Indicação")
    .otherwise("Outro")
)

df = df.withColumn(
    "ramal",
    regexp_extract(col("telefone"), r"(?i)r\.?\s*(\d+)", 1)
)

df = df.withColumn(
    "ramal",
    when(col("ramal") == "", None).otherwise(col("ramal"))
)

df = df.withColumn(
    "telefone_limpo",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(col("telefone"), r"(?i)r\.?\s*\d+", ""),
                r"\D", ""
            ),
            r"^55(?=\d{10,11}$)", ""
        )
    )
)

df = df.withColumn(
    "telefone_formatado",
    when(length(col("telefone_limpo")) == 10,
        concat(
            lit("("), substring("telefone_limpo", 1, 2), lit(") "),
            substring("telefone_limpo", 3, 4), lit("-"),
            substring("telefone_limpo", 7, 4)
        )
    ).when(length(col("telefone_limpo")) == 11,
        concat(
            lit("("), substring("telefone_limpo", 1, 2), lit(") "),
            substring("telefone_limpo", 3, 5), lit("-"),
            substring("telefone_limpo", 8, 4)
        )
    ).otherwise(None) 
)

df = df.withColumn(
    "email_tratado",
    lower(trim(col("email")))
)

df = df.withColumn(
    "email_tratado",
    when(
        col("email_tratado").isNotNull() & (~col("email_tratado").contains("@")),
        regexp_replace(
            col("email_tratado"),
            r"(gmail|yahoo|hotmail|outlook|uol)",
            r"@\1"
        )
    ).otherwise(col("email_tratado"))
)

df = df.withColumn(
    "email_tratado",
    regexp_replace(col("email_tratado"), r"@{2,}", "@")
)

df = df.select( 
    col("id_cliente").cast("string"), 
    col("nome").cast("string").alias("nome_cliente"),
    col("sobrenome").cast("string").alias("sobrenome_cliente"), 
    col("email_tratado").alias("email_cliente"),
    col("telefone_formatado").alias("telefone_cliente"),
    col("ramal").alias("ramal_cliente"),
    col("genero").cast("string").alias("genero_cliente"), 
    col("data_nascimento").cast("date").alias("data_nascimento_cliente"),
    col("data_cadastro").cast("date").alias("data_cadastro_cliente"),
    col("endereco").cast("string").alias("endereco_cliente"),
    col("cidade").cast("string").alias("cidade_cliente"), 
    col("estado").cast("string").alias("estado_cliente"), 
    col("pais").cast("string").alias("pais_cliente"),
    col("origem").alias("origem_cliente")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes")

Dispositivos por cliente

In [0]:
df = spark.table("bronze.clientes")

df_dispositivo = df.select(
    col("id_cliente").cast("string"),
    explode(
        split(col("device_ids"), ";")
    ).alias("device_id")
)

df_dispositivo = df_dispositivo.withColumn(
    "device_id",
    trim(col("device_id"))
)

df_dispositivo = df_dispositivo.filter(col("device_id") != "")

df_dispositivo.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes_dispositivo")

Pedidos

In [0]:
df = spark.table("bronze.pedidos")

df = df.select( 
    col("id_pedido").cast("string"), 
    col("id_cliente").cast("string"),
    col("id_produto").cast("string"),
    col("valor_pedido").cast("decimal(10,2)"), # tem conversões a serem feitas
    col("data_pedido").cast("date"), # tem conversões a serem feitas
    col("metodo_pagamento").cast("string"), # tem conversões a serem feitas
    col("status").cast("string").alias("status_pedido"), # tem conversões a serem feitas
    col("quantidade").cast("int").alias("quantidade_produto") # tem conversões a serem feitas
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.pedidos")

Tickets Suporte

In [0]:
df = spark.table("bronze.suporte_tickets")

df = df.withColumn(
    "tipo_problema",
    when(
        lower(trim(col("tipo_problema"))).isin(
            "pro","produto","p3oduto","produto","prod","product","produto"
        ), "Produto"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "pag","pagamento","p4gamento","pay","payment"
        ), "Pagamento"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "entrega","3ntrega","entr","ent","delay","del"
        ), "Entrega"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "reembolso","reemb","r3embolso","refund","ref","reembolso"
        ), "Reembolso"
    ).otherwise("Outro")
)

df = df.select( 
    col("ticket_id").cast("string"), 
    col("id_cliente").cast("string"),
    col("id_pedido").cast("string"),
    col("tipo_problema"),
    col("data_abertura").cast("timestamp"),
    col("data_resolucao").cast("timestamp"),
    col("tempo_resolucao_horas").cast("decimal(10,2)"),
    col("agente_suporte").cast("string"),
    col("nota_avaliacao").cast("int").alias("nota_avaliacao_problema")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.suporte_tickets")